# Diagnosing the ST-GNN advantage (Colab)

Uses original labels and the already audited Test 2 checkpoints. It compares the same flows for ST-GNN, Static GNN, and Edge GRU without assigning attack stages, tactics, or IPs. It also performs an **exploratory intervention**: resetting ST-GNN memory before every window without retraining. This tests checkpoint sensitivity to history, but does not establish a general causal contribution. Existing artifacts are not modified.


In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('torch_geometric') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/nids-mitre')
RESULTS = DRIVE_ROOT / 'results_earlystopping'
GRAPH_ROOT = DRIVE_ROOT / 'dataset_processed_thu0103'
CLONE_DIR = Path('/content/temporalgnn-nids-mechanism')
BRANCH = 'feat/fair-comparison-experiments'
EXPECTED_COMMIT = '2262b53d155f9e8cf5221516f47e4170e6cd0d6e'
assert RESULTS.is_dir() and (GRAPH_ROOT / 'test2').is_dir()
if not CLONE_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/tatipar/temporalgnn-nids.git', str(CLONE_DIR)], check=True)
commit = subprocess.check_output(['git', '-C', str(CLONE_DIR), 'rev-parse', 'HEAD'], text=True).strip()
assert commit == EXPECTED_COMMIT, f'Code commit changed: {commit}; expected {EXPECTED_COMMIT}'
sys.path.insert(0, str(CLONE_DIR / 'code/python'))
print('Pinned code:', commit)


In [ ]:
import gc, json
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import torch
from torch_geometric.loader import DataLoader
from sklearn.metrics import average_precision_score
from IPython.display import display
from utils.datasets import NF_IDS_Dataset
from utils.models import ST_GNN_Identity, StaticGNN_Identity, EdgeGRU_Baseline_NoX
from utils.metrics import calculate_metrics_gnn

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PARAMS = dict(node_dim=16, edge_dim=32, hidden_dim=32, dropout=0.2,
              output_bias_init=-2.9968)
SPECS = {
    'ST-GNN': ('ST_GNN_BiasOn_robust_Identity_clone', ST_GNN_Identity, 42, True),
    'Static GNN': ('StaticGNN_BiasOn_robust_Identity', StaticGNN_Identity, 42, False),
    'Edge GRU': ('EdgeGRU_NoX_BiasOn', EdgeGRU_Baseline_NoX, 99, True),
}
DATASET = NF_IDS_Dataset(root_dir=str(GRAPH_ROOT), split='test2')
assert len(DATASET) == 2822, f'Expected 2,822 graphs; found {len(DATASET)}'
print('Device:', DEVICE, '| graph files:', len(DATASET))

def artifact(name):
    exp, cls, expected_seed, temporal = SPECS[name]
    old_path = RESULTS / 'test2_results' / f'test2_metrics_{exp}.csv'
    old = pd.read_csv(old_path)
    assert len(old) == 1 and int(old.iloc[0]['seed']) == expected_seed
    old = old.iloc[0]
    run_id = str(old['run_id'])
    hits = list((RESULTS / 'saved_models' / exp).glob(f'{run_id}_*.pth'))
    assert len(hits) == 1, f'Ambiguous checkpoint for {run_id}: {hits}'
    with np.load(RESULTS / 'logs' / exp / f'thresholds_{exp}.npz') as ths:
        th = float(ths[f'seed_{expected_seed}'])
    return dict(name=name, experiment=exp, cls=cls, temporal=temporal,
                seed=expected_seed, run_id=run_id, checkpoint=hits[0],
                threshold=th, old=old)

ARTIFACTS = {name: artifact(name) for name in SPECS}
display(pd.DataFrame([
    {k: v for k, v in a.items() if k in ('name','experiment','seed','run_id','threshold')}
    for a in ARTIFACTS.values()
]))


## 1. Aligned inference and memory sensitivity

The first three passes follow the original protocol. The fourth uses **the same ST-GNN checkpoint** but clears its memory before every window. Its AP is informative; F1 at the original threshold is descriptive because the score distribution changes.


In [ ]:
@torch.no_grad()
def score_model(name, reset_each_window=False, capture_meta=False):
    a = ARTIFACTS[name]
    model = a['cls'](**PARAMS).to(DEVICE)
    state = torch.load(a['checkpoint'], map_location=DEVICE, weights_only=True)
    model.load_state_dict(state, strict=True)
    model.eval()
    if hasattr(model, 'reset_memory'):
        model.reset_memory()
    loader = DataLoader(DATASET, batch_size=1, shuffle=False, num_workers=0)
    ys, ps, wins = [], [], []
    meta = {'source_gid':[], 'out_degree':[], 'prior_windows':[]} if capture_meta else None
    seen = {}
    for batch_idx, data in enumerate(loader):
        if data.x.shape[0] == 0:
            continue
        window_id = int(Path(DATASET.files[batch_idx]).stem.split('_')[1])
        y = data.y.cpu().numpy().astype(np.uint8)
        if capture_meta:
            src = data.edge_index[0].cpu().numpy()
            gids = data.global_node_ids.cpu().numpy()
            degrees = np.bincount(src, minlength=len(gids))[src].astype(np.int32)
            prior_nodes = np.fromiter((seen.get(int(g), 0) for g in gids),
                                      dtype=np.int32, count=len(gids))
            meta['source_gid'].append(gids[src].astype(np.int64))
            meta['out_degree'].append(degrees)
            meta['prior_windows'].append(prior_nodes[src])
            for g in gids:
                key = int(g)
                seen[key] = seen.get(key, 0) + 1
        if reset_each_window:
            model.reset_memory()
        data = data.to(DEVICE)
        if a['temporal']:
            logits = model(data.x, data.edge_index, data.edge_attr, data.global_node_ids)
        else:
            logits = model(data.x, data.edge_index, data.edge_attr)
        p = torch.sigmoid(logits.view(-1)).cpu().numpy().astype(np.float32)
        ys.append(y)
        ps.append(p)
        wins.append(np.full(len(y), window_id, dtype=np.int32))
    y = np.concatenate(ys)
    p = np.concatenate(ps)
    win = np.concatenate(wins)
    metrics = calculate_metrics_gnn(y, p, prob_threshold=a['threshold'])
    if meta is not None:
        meta = {k: np.concatenate(chunks) for k, chunks in meta.items()}
    del model, state
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return dict(y=y, p=p, window=win, pred=p > a['threshold'],
                metrics=metrics, meta=meta)

RUNS = {}
for name in SPECS:
    print('Inference:', name)
    result = score_model(name, capture_meta=(name == 'ST-GNN'))
    a = ARTIFACTS[name]
    for metric in ('F1', 'AUC-PR', 'Precision', 'Recall', 'TP', 'FP', 'TN', 'FN'):
        new, old = result['metrics'][metric], a['old'][metric]
        assert np.isclose(new, old, rtol=1e-4, atol=1e-4), (
            f'{name}: {metric}={new}, earlier CSV={old}')
    if RUNS:
        assert np.array_equal(result['y'], RUNS['ST-GNN']['y'])
        assert np.array_equal(result['window'], RUNS['ST-GNN']['window'])
    RUNS[name] = result
    print('  Reproduced Test 2:', len(result['y']), 'flows; AP',
          round(result['metrics']['AUC-PR'], 4), 'F1', round(result['metrics']['F1'], 4))
print('Inference: ST-GNN with memory reset before every window')
RUNS['ST-GNN with reset memory (intervention)'] = score_model('ST-GNN', reset_each_window=True)
assert np.array_equal(RUNS['ST-GNN with reset memory (intervention)']['y'], RUNS['ST-GNN']['y'])
assert np.array_equal(RUNS['ST-GNN with reset memory (intervention)']['window'], RUNS['ST-GNN']['window'])


## 2. Where ST-GNN gains

The comparison pairs predictions for **the same flow** across models and separates exclusive true positives from exclusive false positives. It then stratifies by the source's outgoing degree **in the current window** and by the number of earlier windows in which that node appeared. These are descriptive associations: current volume, history, and other attributes may be correlated.


In [ ]:
y = RUNS['ST-GNN']['y']
meta = RUNS['ST-GNN']['meta']
rows = []
for name, r in RUNS.items():
    m = r['metrics']
    rows.append(dict(model=name, flows=len(y), AP=m['AUC-PR'], F1=m['F1'],
                     precision=m['Precision'], recall=m['Recall'], FPR=m['FPR'],
                     TP=m['TP'], FP=m['FP']))
summary = pd.DataFrame(rows)
display(summary.round(4))

st = RUNS['ST-GNN']['pred']
pair_rows = []
for name in ('Static GNN', 'Edge GRU'):
    other = RUNS[name]['pred']
    pos, neg = y == 1, y == 0
    only_st_tp = pos & st & ~other
    only_other_tp = pos & ~st & other
    only_st_fp = neg & st & ~other
    only_other_fp = neg & ~st & other
    ids = pd.Series(meta['source_gid'][only_st_tp]).value_counts()
    pair_rows.append(dict(comparator=name,
                          both_TP=int((pos & st & other).sum()),
                          ST_only_TP=int(only_st_tp.sum()),
                          other_only_TP=int(only_other_tp.sum()),
                          neither_TP=int((pos & ~st & ~other).sum()),
                          ST_only_FP=int(only_st_fp.sum()),
                          other_only_FP=int(only_other_fp.sum()),
                          ST_only_TP_top1_source_share=(float(ids.iloc[0] / ids.sum())
                                                        if len(ids) else np.nan),
                          ST_only_TP_top10_source_share=(float(ids.iloc[:10].sum() / ids.sum())
                                                         if len(ids) else np.nan)))
pairs = pd.DataFrame(pair_rows)
display(pairs.round(4))

prior = meta['prior_windows']
degree = meta['out_degree']
prior_bins = np.select([prior == 0, prior <= 2, prior <= 9],
                       ['0', '1-2', '3-9'], default='10+')
degree_bins = np.select([degree == 1, degree <= 4, degree <= 19],
                        ['1', '2-4', '5-19'], default='20+')

def group_report(values, variable, categories):
    rows = []
    for category in categories:
        ix = values == category
        npos = int((y[ix] == 1).sum())
        nneg = int((y[ix] == 0).sum())
        row = dict(variable=variable, group=category, flows=int(ix.sum()),
                   attacks=npos, benign=nneg)
        for name in ('ST-GNN', 'Static GNN', 'Edge GRU'):
            pred = RUNS[name]['pred'][ix]
            prob = RUNS[name]['p'][ix]
            row[f'{name}_recall'] = float(pred[y[ix] == 1].mean()) if npos else np.nan
            row[f'{name}_FPR'] = float(pred[y[ix] == 0].mean()) if nneg else np.nan
            row[f'{name}_AP'] = (float(average_precision_score(y[ix], prob))
                                  if npos and nneg else np.nan)
        rows.append(row)
    return rows

prior_categories = ['0','1-2','3-9','10+']
degree_categories = ['1','2-4','5-19','20+']
joint_bins = np.char.add(np.char.add(prior_bins, ' / '), degree_bins)
joint_categories = [p + ' / ' + d for p in prior_categories for d in degree_categories]
grouped = pd.DataFrame(
    group_report(prior_bins, 'previous source windows', prior_categories)
    + group_report(degree_bins, 'outgoing edges in current window', degree_categories)
    + group_report(joint_bins, 'history / outgoing degree', joint_categories)
)
display(grouped.round(4))


## 3. Interpretation and record

If resetting memory reduces AP, this checkpoint **uses** historical state; it does not show that a retrained temporal architecture would outperform a non-temporal one. A gain over Static GNN among previously seen sources supports a historical-context hypothesis; a gain limited to high-degree sources supports an instantaneous-activity hypothesis. Check source concentration before attributing a pattern from one or two hosts to an entire campaign. A causal claim would require retraining variants without memory and without spatial message passing under the same seeds, data, and validation-selection protocol. These comparisons are diagnostic.


In [ ]:
out_dir = DRIVE_ROOT / 'results_comparison' / 'stgnn_gain_audit'
out_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
summary.to_csv(out_dir / f'summary_{stamp}.csv', index=False)
pairs.to_csv(out_dir / f'paired_{stamp}.csv', index=False)
grouped.to_csv(out_dir / f'groups_{stamp}.csv', index=False)
manifest = {
    'code_commit': EXPECTED_COMMIT,
    'graph_root': str(GRAPH_ROOT / 'test2'),
    'original_labels': True,
    'threshold_operator': '>',
    'intervention': 'reset ST-GNN node memory before every nonempty graph window; no retraining',
    'runs': {name: {k: str(a[k]) for k in ('experiment','seed','run_id','checkpoint','threshold')}
             for name, a in ARTIFACTS.items()},
}
(out_dir / f'manifest_{stamp}.json').write_text(json.dumps(manifest, indent=2))
print('Results:', out_dir, '| suffix:', stamp)
